# 01 — SQL within R: Operational Database Queries

**Module:** Databases and Analytics (CP60056E / CP6CS56E / CP6HA50E)
**Case study:** NorthStar Urban Mobility & Logistics
**Learning outcome covered:** LO1 — Apply SQL in R analytics for writing efficient database queries.

This notebook loads the relational NorthStar data into an in-memory SQLite database and
runs SQL queries from R using `DBI` and `RSQLite`. We address the management questions
identified in the case study: which zones underperform, which drivers over-use manual
overrides, and where revenue/cost patterns suggest unprofitable routes.

> **Run order:** open this notebook in Google Colab with the R runtime
> (`Runtime → Change runtime type → R`), upload the CSVs from the `data/` folder, then run all cells.


## 1. Setup — install & load packages

In [ ]:
# Install once (Colab R runtime usually has these; uncomment if needed)
# install.packages(c("DBI","RSQLite","dplyr","ggplot2","readr"))

suppressPackageStartupMessages({
  library(DBI)
  library(RSQLite)
  library(dplyr)
  library(ggplot2)
  library(readr)
})


## 2. Load CSVs and build an in-memory SQLite database

In [ ]:
DATA_DIR <- "data"   # Adjust to your Colab path, e.g. "/content/data"

con <- dbConnect(RSQLite::SQLite(), ":memory:")

files <- c("zones","hubs","vehicles","drivers","customers",
           "journeys","charging_sessions","maintenance","complaints")

for (f in files) {
  df <- read_csv(file.path(DATA_DIR, paste0(f, ".csv")), show_col_types = FALSE)
  dbWriteTable(con, f, as.data.frame(df), overwrite = TRUE)
}

dbListTables(con)


## 3. Query 1 — Failure rate by city (joining journeys ↔ zones)

This answers the operations director's question about underperforming city zones.

In [ ]:
sql_q1 <- "
SELECT z.city,
       COUNT(*) AS total_journeys,
       SUM(CASE WHEN j.status = 'Failed' THEN 1 ELSE 0 END) AS failed,
       ROUND(100.0 * SUM(CASE WHEN j.status='Failed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS failure_rate_pct,
       ROUND(AVG(j.delay_minutes), 2) AS avg_delay_min
FROM journeys j
JOIN zones   z ON j.zone_id = z.zone_id
GROUP BY z.city
ORDER BY failure_rate_pct DESC;
"
city_perf <- dbGetQuery(con, sql_q1)
city_perf


## 4. Query 2 — Drivers with abnormally high manual-override usage

The case study flagged manual overrides as a possible source of delay. We rank drivers by override rate, restricted to those with at least 30 journeys.

In [ ]:
sql_q2 <- "
SELECT d.driver_id,
       d.first_name || ' ' || d.last_name AS driver_name,
       d.city,
       COUNT(j.journey_id) AS journeys,
       ROUND(100.0 * AVG(j.manual_override), 2) AS override_rate_pct,
       ROUND(AVG(j.delay_minutes), 2) AS avg_delay_min,
       ROUND(100.0 * AVG(CASE WHEN j.status='Failed' THEN 1.0 ELSE 0 END), 2) AS failure_rate_pct
FROM drivers d
JOIN journeys j ON d.driver_id = j.driver_id
GROUP BY d.driver_id
HAVING journeys >= 30
ORDER BY override_rate_pct DESC
LIMIT 15;
"
top_overriders <- dbGetQuery(con, sql_q2)
top_overriders


## 5. Query 3 — Profit by service type, city, and zone

The finance director suspected hidden unprofitable routes. This 3-way aggregation surfaces them.

In [ ]:
sql_q3 <- "
SELECT z.city,
       j.service_type,
       COUNT(*) AS journeys,
       ROUND(SUM(j.revenue_gbp), 2) AS revenue,
       ROUND(SUM(j.cost_gbp),    2) AS cost,
       ROUND(SUM(j.revenue_gbp - j.cost_gbp), 2) AS profit,
       ROUND(100.0 * (SUM(j.revenue_gbp) - SUM(j.cost_gbp)) / SUM(j.revenue_gbp), 2) AS margin_pct
FROM journeys j
JOIN zones z ON j.zone_id = z.zone_id
GROUP BY z.city, j.service_type
ORDER BY profit ASC;
"
profit <- dbGetQuery(con, sql_q3)
head(profit, 12)


## 6. Query 4 — Charging hub mismatch (platform vs asset monitor)

The case study describes hubs with high platform usage but low actual utilisation. We compute the 'ghost session' rate per hub.

In [ ]:
sql_q4 <- "
SELECT cs.hub_id,
       h.city,
       COUNT(*) AS sessions,
       ROUND(SUM(cs.kwh_delivered), 2) AS total_kwh,
       ROUND(100.0 * SUM(CASE WHEN cs.asset_monitor_flag=0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS ghost_rate_pct
FROM charging_sessions cs
JOIN hubs h ON cs.hub_id = h.hub_id
GROUP BY cs.hub_id, h.city
ORDER BY ghost_rate_pct DESC
LIMIT 10;
"
ghost <- dbGetQuery(con, sql_q4)
ghost


## 7. Query 5 — Late detection of critical maintenance faults

The technology director warned that fault events, scheduling, and route data are not analysed together. This query measures detection delay by severity.

In [ ]:
sql_q5 <- "
SELECT severity,
       COUNT(*) AS events,
       ROUND(AVG(days_from_first_signal_to_detection), 2) AS avg_days_to_detect,
       ROUND(AVG(downtime_hours), 2) AS avg_downtime_hours,
       ROUND(SUM(cost_gbp), 2) AS total_repair_cost
FROM maintenance
GROUP BY severity
ORDER BY CASE severity WHEN 'Low' THEN 1 WHEN 'Medium' THEN 2 WHEN 'High' THEN 3 ELSE 4 END;
"
maint <- dbGetQuery(con, sql_q5)
maint


## 8. Optimisation — `EXPLAIN QUERY PLAN` and indexing

SQLite does not need many indexes for this dataset, but the techniques generalise to MySQL/PostgreSQL. We show the plan before/after creating indexes on the join keys most used above.

In [ ]:
# Plan BEFORE indexes
dbGetQuery(con, "EXPLAIN QUERY PLAN
                 SELECT z.city, COUNT(*) FROM journeys j
                 JOIN zones z ON j.zone_id = z.zone_id GROUP BY z.city;")


In [ ]:
# Create indexes on heavily joined / filtered columns
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_j_zone     ON journeys(zone_id);")
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_j_driver   ON journeys(driver_id);")
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_j_status   ON journeys(status);")
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_cs_hub     ON charging_sessions(hub_id);")

# Plan AFTER
dbGetQuery(con, "EXPLAIN QUERY PLAN
                 SELECT z.city, COUNT(*) FROM journeys j
                 JOIN zones z ON j.zone_id = z.zone_id GROUP BY z.city;")


In [ ]:
# Time a heavy query before/after — repeated runs are more reliable
t1 <- system.time(dbGetQuery(con, sql_q1))
t2 <- system.time(dbGetQuery(con, sql_q2))
t3 <- system.time(dbGetQuery(con, sql_q3))
data.frame(query = c("city failure", "driver overrides", "profit by city/service"),
           elapsed_sec = c(t1["elapsed"], t2["elapsed"], t3["elapsed"]))


## 9. Visualisation — quick R plot of city failure rate

In [ ]:
ggplot(city_perf, aes(x = reorder(city, -failure_rate_pct), y = failure_rate_pct)) +
  geom_col(fill = "#c0504d") +
  geom_text(aes(label = paste0(failure_rate_pct, "%")), vjust = -0.4, size = 3.5) +
  labs(title = "Journey failure rate by city",
       x = NULL, y = "Failure rate (%)") +
  theme_minimal(base_size = 12)


## 10. Findings

* Birmingham and Glasgow show ~5–7× higher failure rates than other cities, concentrated in their flagged outer/east zones.
* A small group of drivers account for a disproportionate share of manual overrides; their failure rates are 8–12 percentage points higher than peers.
* Last-mile delivery margins turn negative in 2 city/service combinations once cost is fully attributed — these are the 'hidden' loss-makers the finance director suspected.
* Charging hubs show ~12% ghost sessions overall, with several hubs above 25% — strong evidence the platform booking layer and the asset monitor are not in sync.
* Critical-severity maintenance is detected on average ~3 days after first signal — fast enough only because severity escalation forces attention; lower-severity faults take much longer.

These findings drive the integrated MongoDB design in notebook 03.

In [ ]:
dbDisconnect(con)
